## Import

In [1]:
import warnings
warnings.filterwarnings("ignore")
import os
import time
import numpy as np
import scipy.linalg as linalg
import torch
import torch.nn as nn
import torchvision as tv
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torch.nn.functional as F
import torch.optim as optim
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
#import preprocess


device = 'cuda:0'

%load_ext autoreload
%autoreload 2

## Dataset

In [10]:
from datasets.Student_t import MultivariateStudentT

d = 32
dim_x = dim_y = d//2
rho = 0.8
df = 1
mean = np.zeros(dim_x + dim_y)
V = np.eye(dim_x)*rho
dispersion = np.eye(dim_x + dim_y)
dispersion[0:dim_x, :][:, dim_x:] = V
dispersion[dim_x:, :][:, 0:dim_x] = V


dataset = MultivariateStudentT(
        dim_x=dim_x,
        dim_y=dim_y,
        mean=mean,
        dispersion=dispersion,
        df=df)

X, Y = dataset.sample(10000)
X, Y = torch.Tensor(X).float().to(device), torch.Tensor(Y).float().to(device)

print(f'MI is {dataset.mutual_information(): 0.2f}')
print('X', X.shape, 'Y', Y.shape)

MI is  9.52
X torch.Size([10000, 16]) Y torch.Size([10000, 16])


## Hyperaparams

In [6]:
class Hyperparams(object):
    def __init__(self): 
        self.critic = 'neural'                # ('neural', 'quadratic')
        self.lr = 5e-4
        self.bs = 500
        self.n_bridges = 4
        self.wd = 1e-5
        
hyperparams=Hyperparams()

architecture_critic = [d, 500, 500, 500, 1]
architecture_encode = [d//2, 200, 200, d//2]

## MI estimate

In [7]:
## Mutual information neural diffusion estimate (MINDE)
from estimators.MINDE import MINDE

hyperparams.t_patience = 500
hyperparams.dim = d//2
hyperparams.device = device
hyperparams.importance_sampling = True

estimator = MINDE(None, None, None, hyperparams)

estimator.to(device)
estimator.learn(X, Y)

print('true MI:', dataset.mutual_information())
print('est MI:', estimator.MI(X, Y))

finished: t= 0 loss= 0.9994271993637085 loss val= 1.0001064538955688 best val loss= 1.0001064538955688 best t= 0
finished: t= 76 loss= 0.6088359951972961 loss val= 0.6191346049308777 best val loss= 0.5727471709251404 best t= 75
finished: t= 152 loss= 0.5927891135215759 loss val= 0.5418891906738281 best val loss= 0.5418891906738281 best t= 152
finished: t= 228 loss= 0.5640163421630859 loss val= 0.5928633213043213 best val loss= 0.5407195091247559 best t= 175
finished: t= 304 loss= 0.6185488104820251 loss val= 0.5598042011260986 best val loss= 0.5347765684127808 best t= 270
finished: t= 380 loss= 0.5380341410636902 loss val= 0.5551213026046753 best val loss= 0.5292068719863892 best t= 307
finished: t= 456 loss= 0.5328512787818909 loss val= 0.5583264231681824 best val loss= 0.5199465751647949 best t= 430
finished: t= 532 loss= 0.5550535917282104 loss val= 0.5287967920303345 best val loss= 0.5199465751647949 best t= 430
finished: t= 608 loss= 0.5499024987220764 loss val= 0.5514392256736755

In [11]:
## Mutual information neural estimate (MINE)
from estimators.MINE import MINE

estimator = MINE(None, None, architecture_critic, hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('true MI:', dataset.mutual_information())
print('est MI:', estimator.MI(X, Y))

finished: t= 0 loss= 73.04354858398438 loss val= 237.21078491210938 best val loss= 237.21078491210938 best t= 0
finished: t= 76 loss= 2.4219236373901367 loss val= 3.350947141647339 best val loss= 1.2304739952087402 best t= 71
finished: t= 152 loss= 7.492395877838135 loss val= 7.131320953369141 best val loss= 0.45309680700302124 best t= 135
finished: t= 228 loss= 5.330896377563477 loss val= 5.758667945861816 best val loss= 0.45309680700302124 best t= 135


true MI: 9.517696203706947
est MI: 1.0890768766403198


In [13]:
## Vector copula estimation
from estimators.VCE import VCE

estimator = VCE(None, None, architecture_critic, hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('true MI:', dataset.mutual_information())
print('est MI:', estimator.MI(X, Y))


K components= 5 copula transform= True
nde type: FM
finished: t= 0 loss= 3726.71044921875 loss val= 4712.7216796875 best val loss= 4712.7216796875 best t= 0
finished: t= 126 loss= 3887.507568359375 loss val= 4088.09912109375 best val loss= 3546.75732421875 best t= 60


finished: t= 0 loss= 75.63214111328125 loss val= 5324.1455078125 best val loss= 5324.1455078125 best t= 0
finished: t= 126 loss= 40458.25390625 loss val= 4909.70556640625 best val loss= 4575.70751953125 best t= 41


finished: t= 0 loss= 125.61051177978516 loss val= 124.11595916748047 best val loss= 124.11595916748047 best t= 0
finished: t= 101 loss= 30.719884872436523 loss val= 30.823535919189453 best val loss= 30.823535919189453 best t= 101
finished: t= 202 loss= 30.455236434936523 loss val= 30.821685791015625 best val loss= 30.80628204345703 best t= 181
finished: t= 303 loss= 30.25977897644043 loss val= 30.820411682128906 best val loss= 30.804122924804688 best t= 250
finished: t= 404 loss= 30.56600570678711 loss val= 3